In [58]:
import numpy as np

In [59]:
## data preparation
def prepare_data_splits(X, y, train_prop = 0.98, dev_prop = 0.01):
    m = X.shape[0]
    shuffled_indices = np.random.permutation(m)
    X_shuffled = X[shuffled_indices]
    y_shuffled = y[shuffled_indices]

    train_end = int(m * train_prop)
    dev_end = train_end + int(m * dev_prop)

    X_train, y_train = X_shuffled[:train_end], y_shuffled[:train_end]
    X_dev, y_dev = X_shuffled[train_end:dev_end], y_shuffled[train_end:dev_end]
    X_test, y_test =  X_shuffled[dev_end:], y_shuffled[dev_end:]
    return X_train, y_train, X_dev, y_dev, X_test, y_test


In [60]:
##normalization features

def normalization_features(X_train, X_dev, X_test):
    mu = np.mean(X_train, axis = 0, keepdims = True)
    sigma = np.std(X_train, axis = 0, keepdims = True)

    X_train_norm = (X_train - mu) / sigma
    X_dev_norm  = (X_dev - mu) / sigma
    X_test_norm = (X_test - mu) / sigma
    
    return X_train_norm, X_dev_norm, X_test_norm



Input-> Hidden1 (ReLu) -> Hidden2 (ReLu) -> OutPut(Softmax/ Sigmoid)

In [61]:
## initialize parameters naive 

def initialize_parameter_naive(layer_dims):
    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    
    return parameters



In [62]:
## forward propagation
def relu(Z): return np.maximum(0, Z)
def softmax(Z):
    exp_Z = np.exp(Z -np.max(Z, axis = 0, keepdims = True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

def forward_propagation(X_batch, parameters):
    caches = {}
    A = X_batch.T
    caches['A0'] = A
    L = len(parameters) // 2

    for l in range(1, L):
        Z = np.dot(parameters['W' + str(l)], A)  + parameters['b' + str(l)]
        A = relu(Z)

        caches['Z' + str(l)] = Z
        caches['A' + str(l)] = A

    ZL = np.dot(parameters['W' + str(L)], A) + parameters['b' + str(L)]
    AL = softmax(ZL)
    caches['ZL' + str(L)] = ZL
    caches['AL' + str(L)] = AL

    return AL, caches

In [63]:
# compute cost

def compute_cost(AL, Y_batch):
    m = Y_batch.shape[0]
    Y = Y_batch.T

    # cost = -(1.0 / m) * np.sum(Y * np.log(AL + 1e-8))
    cost = - (1.0 / m) * np.sum(Y * np.log(AL + 1e-8))
    return np.squeeze(cost)
    

In [64]:
## Backward Propagation

def relu_backward(dA, Z):
    dZ = np.array(dA, copy=True)
    dZ[Z<=0] = 0
    return dZ

def backward_propagation(caches, parameters, Y_batch):
    grads = {}
    m = Y_batch.shape[0]
    Y = Y_batch.T
    L = len(parameters) // 2

    dZL = caches['AL' + str(L)] - Y
    grads['dW' + str(L)] = (1.0 / m) * np.dot(dZL, caches['A' + str(L-1)].T)
    grads['db' + str(L)] = (1.0 / m) * np.sum(dZL, axis=1, keepdims=True)

    dAl = np.dot(parameters['W' + str(L)].T, dZL)
    for l in reversed(range(1, L)):
        dZl = relu_backward(dAl, caches['Z' + str(l)])
        grads['dW' + str(l)] = (1.0 / m) * np.dot(dZl, caches['A' + str(l-1)].T)
        grads['db' + str(l)] = (1.0 / m) * np.sum(dZl, axis=1, keepdims=True)
        if l > 1:
            dAl = np.dot(parameters['W' + str(l)].T, dZl)
        
    return grads


In [65]:
## optimization and training loop

def update_parameters_sgd(parameters, grads, lr):
    L = len(parameters) // 2
    for l in range(1, L + 1):
        parameters['W' + str(l)] -= lr * grads['dW' + str(l)]
        parameters['b' + str(l)] -= lr * grads['db' + str(l)]
    return parameters

In [66]:
# create mini batches 
def create_mini_batches(X, y, batch_size = 64, seed = 42):
    np.random.seed(seed)
    m = X.shape[0]
    mini_batches = []

    permutation = list(np.random.permutation(m))
    shuffled_X = X[permutation, :]
    shuffled_y = y[permutation]

    num_complete_batches = m // batch_size
    for k in range(num_complete_batches):
        mini_batch_X = shuffled_X[k * batch_size: (k + 1) * batch_size, :]
        mini_batch_y = shuffled_y[k * batch_size: (k + 1) * batch_size]
        mini_batches.append((mini_batch_X, mini_batch_y))
    
    if m % batch_size != 0:
        mini_batch_X = shuffled_X[num_complete_batches * batch_size :, :]
        mini_batch_y = shuffled_y[num_complete_batches * batch_size :]
        mini_batches.append((mini_batch_X, mini_batch_y))

    return mini_batches



In [67]:


# def create_mini_batches(X, y, batch_size=64, seed=42):
#     np.random.seed(seed)
#     m = X.shape[0]
#     mini_batches = []
    
#     # Step 1: Shuffle X and y consistently
#     permutation = list(np.random.permutation(m))
#     shuffled_X = X[permutation, :]
#     shuffled_y = y[permutation]
    
#     # Step 2: Partition into mini-batches
#     num_complete_batches = m // batch_size
#     for k in range(num_complete_batches):
#         mini_batch_X = shuffled_X[k * batch_size : (k + 1) * batch_size, :]
#         mini_batch_y = shuffled_y[k * batch_size : (k + 1) * batch_size]
#         mini_batches.append((mini_batch_X, mini_batch_y))
        
#     # Handling the end case (residual batch if m is not divisible by batch_size)
#     if m % batch_size != 0:
#         mini_batch_X = shuffled_X[num_complete_batches * batch_size :, :]
#         mini_batch_y = shuffled_y[num_complete_batches * batch_size :]
#         mini_batches.append((mini_batch_X, mini_batch_y))
        
#     return mini_batches

In [68]:
## train model phase1

def train_model_phase1(X_train, y_train, layer_dims, epochs = 100, batch_size = 64, lr = 0.01):
    parameters = initialize_parameter_naive(layer_dims)

    for epoch in range(epochs):
        mini_batches = create_mini_batches(X_train, y_train, batch_size, seed = epoch)
        epoch_cost = 0

        for X_batch, y_batch in mini_batches:
            AL, caches = forward_propagation(X_batch, parameters)
            batch_cost = compute_cost(AL, y_batch)
            epoch_cost += batch_cost * (X_batch.shape[0] / X_train.shape[0])
            grads = backward_propagation(caches, parameters, y_batch)
            parameters = update_parameters_sgd(parameters, grads, lr)

        for epoch in range(epochs): 
            print(f"Epoch {epoch} -> Training Cost: {epoch_cost:.4f}")
        
    return parameters



In [69]:
#main function to test the training loop

if __name__ == "__main__":
    print("Generating Synthetic dataset:")
    X_dummy = np.random.randn(2000, 3072)
    y_dummy = np.eye(3)[np.random.randint(0, 3, size=(2000,))]

    layer_dimensions = [3072, 128, 64, 3]
    
    trained_params = train_model_phase1(
        X_dummy, 
        y_dummy, 
        layer_dims=layer_dimensions, 
        epochs=50, 
        batch_size=64, 
        lr=0.05
    )
    print("Execution finished successfully!")


Generating Synthetic dataset:
Epoch 0 -> Training Cost: 1.0987
Epoch 1 -> Training Cost: 1.0987
Epoch 2 -> Training Cost: 1.0987
Epoch 3 -> Training Cost: 1.0987
Epoch 4 -> Training Cost: 1.0987
Epoch 5 -> Training Cost: 1.0987
Epoch 6 -> Training Cost: 1.0987
Epoch 7 -> Training Cost: 1.0987
Epoch 8 -> Training Cost: 1.0987
Epoch 9 -> Training Cost: 1.0987
Epoch 10 -> Training Cost: 1.0987
Epoch 11 -> Training Cost: 1.0987
Epoch 12 -> Training Cost: 1.0987
Epoch 13 -> Training Cost: 1.0987
Epoch 14 -> Training Cost: 1.0987
Epoch 15 -> Training Cost: 1.0987
Epoch 16 -> Training Cost: 1.0987
Epoch 17 -> Training Cost: 1.0987
Epoch 18 -> Training Cost: 1.0987
Epoch 19 -> Training Cost: 1.0987
Epoch 20 -> Training Cost: 1.0987
Epoch 21 -> Training Cost: 1.0987
Epoch 22 -> Training Cost: 1.0987
Epoch 23 -> Training Cost: 1.0987
Epoch 24 -> Training Cost: 1.0987
Epoch 25 -> Training Cost: 1.0987
Epoch 26 -> Training Cost: 1.0987
Epoch 27 -> Training Cost: 1.0987
Epoch 28 -> Training Cost: 1

Phase 2 The Stabilizers

In [70]:
## initializaton parameters he

def initialize_parameters_he(layer_dims):
    np.random.seed(42)
    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        scale = np.sqrt(2.0 / layer_dims[l-1])
        parameters['W' + str(l)]  = np.random.randn(layer_dims[l], layer_dims[l - 1]) * scale
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

In [71]:
def dictionary_to_vector(parameters, layer_dims, is_gradient=False):
    """
    Flattens parameters/gradients in layer order:
    W1, b1, W2, b2, ...
    or
    dW1, db1, dW2, db2, ...
    """
    vector_parts = []
    prefix = "d" if is_gradient else ""

    for l in range(1, len(layer_dims)):
        vector_parts.append(parameters[f"{prefix}W{l}"].reshape(-1, 1))
        vector_parts.append(parameters[f"{prefix}b{l}"].reshape(-1, 1))

    return np.concatenate(vector_parts, axis=0)


def vector_to_dictionary(vector, layer_dims):
    """Reconstructs parameters in the same W1, b1, W2, b2 order."""
    parameters = {}
    current_idx = 0

    for l in range(1, len(layer_dims)):
        w_shape = (layer_dims[l], layer_dims[l - 1])
        w_size = np.prod(w_shape)
        parameters[f"W{l}"] = vector[
            current_idx: current_idx + w_size
        ].reshape(w_shape)
        current_idx += w_size

        b_shape = (layer_dims[l], 1)
        b_size = np.prod(b_shape)
        parameters[f"b{l}"] = vector[
            current_idx: current_idx + b_size
        ].reshape(b_shape)
        current_idx += b_size

    return parameters

In [72]:
## gradient_check
def gradient_check(X, Y, parameters, grads, layer_dims, epsilon=1e-7):
    # Flatten parameters and analytical gradients
    theta = dictionary_to_vector(parameters, layer_dims)
    analytical_grad_vec = dictionary_to_vector(
        grads, layer_dims, is_gradient=True
    )
    
    num_parameters = theta.shape[0]
    grad_approx = np.zeros((num_parameters, 1))
    
    print(f"Starting gradient check across {num_parameters} parameters...")
    
    # Loop over all parameters to compute numerical gradients
    for i in range(num_parameters):
        # Compute J(theta + epsilon)
        theta_plus = np.copy(theta)
        theta_plus[i][0] += epsilon
        params_plus = vector_to_dictionary(theta_plus, layer_dims)
        AL_plus, _ = forward_propagation(X, params_plus) # Uses Phase 1 Forward
        loss_plus = compute_cost(AL_plus, Y)
        
        # Compute J(theta - epsilon)
        theta_minus = np.copy(theta)
        theta_minus[i][0] -= epsilon
        params_minus = vector_to_dictionary(theta_minus, layer_dims)
        AL_minus, _ = forward_propagation(X, params_minus)
        loss_minus = compute_cost(AL_minus, Y)
        
        # Two-sided numerical approximation
        grad_approx[i][0] = (loss_plus - loss_minus) / (2 * epsilon)
        
    # Compute the relative Euclidean distance difference
    numerator = np.linalg.norm(analytical_grad_vec - grad_approx)
    denominator = np.linalg.norm(analytical_grad_vec) + np.linalg.norm(grad_approx)
    difference = numerator / denominator
    
    if difference < 1e-7:
        print(f"✅ Gradient check passed! Difference: {difference:.2e}")
    else:
        print(f"❌ Gradient check FAILED! Difference: {difference:.2e}")
        
    return difference


In [73]:
if __name__ == "__main__":
    print("--- Running Phase 2: Testing Stabilizers ---")
    
    # 1. Setup small setup for checking (Keep sizes small, grad check loops parameter-by-parameter!)
    np.random.seed(1)
    X_tiny = np.random.randn(10, 20) # 10 samples, 20 features
    y_tiny = np.eye(2)[np.random.randint(0, 2, size=(10,))] # 2 target classes
    layer_dimensions = [20, 8, 2] # 20 inputs -> 8 hidden -> 2 outputs
    
    # 2. Use our brand new He Initialization
    print("Initializing weights via He Initialization...")
    parameters = initialize_parameters_he(layer_dimensions)
    
    # 3. Run a single forward and backward pass to get analytical gradients
    AL, caches = forward_propagation(X_tiny, parameters)
    grads = backward_propagation(caches, parameters, y_tiny)
    
    # 4. Run the verification tool
    diff = gradient_check(X_tiny, y_tiny, parameters, grads, layer_dimensions)

--- Running Phase 2: Testing Stabilizers ---
Initializing weights via He Initialization...
Starting gradient check across 186 parameters...
✅ Gradient check passed! Difference: 2.24e-08


phase 3 The Regulizers

In [74]:
## compute_cost_l2

def compute_cost_l2(AL, Y_batch, parameters, lambd):
    m = Y_batch.shape[0]
    Y = Y_batch.T
    
    # Core cross-entropy cost (from Phase 1)
    cross_entropy_cost = - (1.0 / m) * np.sum(Y * np.log(AL + 1e-8))
    
    # Calculate L2 penalty term across all weight matrices
    L = len(parameters) // 2
    l2_sum = 0
    for l in range(1, L + 1):
        l2_sum += np.sum(np.square(parameters['W' + str(l)]))
        
    l2_penalty = (lambd / (2 * m)) * l2_sum
    cost = cross_entropy_cost + l2_penalty
    return np.squeeze(cost)

In [75]:
## forward_propagation_dropout

def forward_propagation_dropout(X_batch, parameters, keep_prob=0.8):
    caches = {}
    A = X_batch.T 
    caches['A0'] = A
    L = len(parameters) // 2
    
    for l in range(1, L):
        Z = np.dot(parameters['W' + str(l)], A) + parameters['b' + str(l)]
        A = np.maximum(0, Z) # ReLU activation
        
        # Inverted Dropout Strategy
        D = np.random.rand(A.shape[0], A.shape[1]) # Step 1: Initialize random matrix
        D = (D < keep_prob).astype(int)            # Step 2: Convert to binary mask (0 or 1)
        A = A * D                                  # Step 3: Shut down hidden units
        A = A / keep_prob                          # Step 4: Scale activations up
        
        caches['Z' + str(l)] = Z
        caches['D' + str(l)] = D
        caches['A' + str(l)] = A
        
    # Output layer (No Dropout on final predictions)
    ZL = np.dot(parameters['W' + str(L)], A) + parameters['b' + str(L)]
    exp_ZL = np.exp(ZL - np.max(ZL, axis=0, keepdims=True))
    AL = exp_ZL / np.sum(exp_ZL, axis=0, keepdims=True)
    caches['ZL' + str(L)] = ZL
    caches['AL' + str(L)] = AL
    
    return AL, caches

In [76]:
## backward_propagation_reg

def backward_propagation_reg(caches, parameters, Y_batch, lambd, keep_prob):
    grads = {}
    L = len(parameters) // 2
    m = Y_batch.shape[0]
    Y = Y_batch.T
    
    # Output layer (Softmax / Cross-Entropy)
    dZL = caches['AL' + str(L)] - Y
    # Add L2 gradient term to dW
    grads['dW' + str(L)] = (1.0 / m) * np.dot(dZL, caches['A' + str(L-1)].T) + (lambd / m) * parameters['W' + str(L)]
    grads['db' + str(L)] = (1.0 / m) * np.sum(dZL, axis=1, keepdims=True)
    
    dAl = np.dot(parameters['W' + str(L)].T, dZL)
    
    # Loop backwards through hidden layers
    for l in reversed(range(1, L)):
        # Apply the exact same dropout mask used in forward propagation
        dAl = dAl * caches['D' + str(l)]
        dAl = dAl / keep_prob # Re-scale backpropagation flow
        
        dZl = np.array(dAl, copy=True)
        dZl[caches['Z' + str(l)] <= 0] = 0 # ReLU backward step
        
        # Add L2 gradient term to hidden dW
        grads['dW' + str(l)] = (1.0 / m) * np.dot(dZl, caches['A' + str(l-1)].T) + (lambd / m) * parameters['W' + str(l)]
        grads['db' + str(l)] = (1.0 / m) * np.sum(dZl, axis=1, keepdims=True)
        
        if l > 1:
            dAl = np.dot(parameters['W' + str(l)].T, dZl)
            
    return grads

def apply_data_augmentation(X_batch, noise_factor=0.05):
    # Add random horizontal Gaussian noise variance to mimic lighting shifts
    noise = np.random.randn(*X_batch.shape) * noise_factor
    return X_batch + noise

In [77]:
# Early stopping implemented in regulizers

def train_model_phase3(X_train, y_train, X_dev, y_dev, layer_dims, epochs=100, batch_size=64, lr=0.01, lambd=0.1, keep_prob=0.8, patience=5):
    # Use Phase 2 He Initialization
    parameters = initialize_parameters_he(layer_dims)
    
    best_dev_loss = float('inf')
    best_params = None
    patience_counter = 0
    
    for epoch in range(epochs):
        mini_batches = create_mini_batches(X_train, y_train, batch_size, seed=epoch)
        
        # --- TRAINING LOOP (WITH REGULARIZATION) ---
        for X_batch, y_batch in mini_batches:
            # Apply data augmentation
            X_batch_aug = apply_data_augmentation(X_batch)
            
            # Forward pass with active Dropout
            AL, caches = forward_propagation_dropout(X_batch_aug, parameters, keep_prob)
            
            # Backward pass calculating L2 and Dropout adjustments
            grads = backward_propagation_reg(caches, parameters, y_batch, lambd, keep_prob)
            
            # Update weights via SGD
            parameters = update_parameters_sgd(parameters, grads, lr)
            
        # --- EVALUATION LOOP (WITHOUT DROPOUT) ---
        # Evaluate on Train Set (keep_prob = 1.0)
        AL_train, _ = forward_propagation_dropout(X_train, parameters, keep_prob=1.0)
        train_loss = compute_cost_l2(AL_train, y_train, parameters, lambd)
        
        # Evaluate on Dev Set (keep_prob = 1.0)
        AL_dev, _ = forward_propagation_dropout(X_dev, parameters, keep_prob=1.0)
        dev_loss = compute_cost_l2(AL_dev, y_dev, parameters, lambd)
        
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:02d} -> Train Loss: {train_loss:.4f} | Dev Loss: {dev_loss:.4f}")
            
        # --- EARLY STOPPING CHECK ---
        if dev_loss < best_dev_loss:
            best_dev_loss = dev_loss
            best_params = parameters.copy() # Cache best working configurations
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"🛑 Early Stopping triggered at epoch {epoch}! Dev loss starting to climb.")
                break
                
    return best_params

In [79]:
if __name__ == "__main__":
    print("--- Running Phase 3: Testing Regularizers ---")
    np.random.seed(42)
    
    # Generate larger synthetic dataset
    X_all = np.random.randn(1000, 50)
    y_all = np.eye(2)[np.random.randint(0, 2, size=(1000,))]
    
    # Use our Phase 1 Data Split & Normalization tools
    X_train, y_train, X_dev, y_dev, _, _ = prepare_data_splits(X_all, y_all, 0.8, 0.1)
    X_train, X_dev, _ = normalization_features(X_train, X_dev, X_dev)
    
    layer_dimensions = [50, 16, 8, 2] # Deeper network
    
    # Run the regularized model
    optimized_params = train_model_phase3(
        X_train, y_train, X_dev, y_dev, 
        layer_dims=layer_dimensions,
        epochs=100, batch_size=32, lr=0.05, 
        lambd=0.7, keep_prob=0.75, patience=8
    )

--- Running Phase 3: Testing Regularizers ---
Epoch 00 -> Train Loss: 0.7393 | Dev Loss: 0.9896
Epoch 05 -> Train Loss: 0.6961 | Dev Loss: 0.8247
Epoch 10 -> Train Loss: 0.6902 | Dev Loss: 0.8036
Epoch 15 -> Train Loss: 0.6836 | Dev Loss: 0.7820
Epoch 20 -> Train Loss: 0.6789 | Dev Loss: 0.7660
Epoch 25 -> Train Loss: 0.6741 | Dev Loss: 0.7557
Epoch 30 -> Train Loss: 0.6693 | Dev Loss: 0.7457
Epoch 35 -> Train Loss: 0.6633 | Dev Loss: 0.7421
Epoch 40 -> Train Loss: 0.6580 | Dev Loss: 0.7404
Epoch 45 -> Train Loss: 0.6496 | Dev Loss: 0.7410
Epoch 50 -> Train Loss: 0.6420 | Dev Loss: 0.7489
🛑 Early Stopping triggered at epoch 51! Dev loss starting to climb.


Phase 4 The TurboChargers


In [80]:
## initialize adams 

def initialize_adam_states(parameters):
    L = len(parameters) // 2
    m = {}
    v = {}
    
    for l in range(1, L + 1):
        m["dW" + str(l)] = np.zeros_like(parameters["W" + str(l)])
        m["db" + str(l)] = np.zeros_like(parameters["b" + str(l)])
        v["dW" + str(l)] = np.zeros_like(parameters["W" + str(l)])
        v["db" + str(l)] = np.zeros_like(parameters["b" + str(l)])
        
    return m, v

In [81]:
## update parameters adam

def update_parameters_adam(parameters, grads, m, v, t, lr, beta1=0.9, beta2=0.999, epsilon=1e-8):
    L = len(parameters) // 2
    
    for l in range(1, L + 1):
        # 1. Momentum tracking (First Moment Estimate)
        m["dW" + str(l)] = beta1 * m["dW" + str(l)] + (1 - beta1) * grads["dW" + str(l)]
        m["db" + str(l)] = beta1 * m["db" + str(l)] + (1 - beta1) * grads["db" + str(l)]
        
        # 2. RMSprop tracking (Second Moment Estimate)
        v["dW" + str(l)] = beta2 * v["dW" + str(l)] + (1 - beta2) * np.square(grads["dW" + str(l)])
        v["db" + str(l)] = beta2 * v["db" + str(l)] + (1 - beta2) * np.square(grads["db" + str(l)])
        
        # 3. Bias Correction (Crucial for early steps when m and v start at 0)
        m_corrected_dW = m["dW" + str(l)] / (1 - beta1 ** t)
        m_corrected_db = m["db" + str(l)] / (1 - beta1 ** t)
        v_corrected_dW = v["dW" + str(l)] / (1 - beta2 ** t)
        v_corrected_db = v["db" + str(l)] / (1 - beta2 ** t)
        
        # 4. Vectorized parameter update step
        parameters["W" + str(l)] -= (lr / (np.sqrt(v_corrected_dW) + epsilon)) * m_corrected_dW
        parameters["b" + str(l)] -= (lr / (np.sqrt(v_corrected_db) + epsilon)) * m_corrected_db
        
    return parameters, m, v

In [82]:
## apply learning rate

def apply_learning_rate_decay(initial_lr, epoch, decay_rate=0.1):
    # Standard time-based decay schedule
    return initial_lr / (1.0 + decay_rate * epoch)

In [83]:
# testing of training loop

def train_model_turbocharged(X_train, y_train, layer_dims, epochs=50, batch_size=64, initial_lr=0.005, lambd=0.1, keep_prob=0.8, decay_rate=0.05):
    # Phase 2: Structural He Initialization
    parameters = initialize_parameters_he(layer_dims)
    
    # Phase 4: Initialize Adam Memory Structs
    m, v = initialize_adam_states(parameters)
    
    t = 0 # Track total global step iterations for Adam bias correction
    
    for epoch in range(epochs):
        # Phase 4: Compute decayed learning rate for this epoch loop
        current_lr = apply_learning_rate_decay(initial_lr, epoch, decay_rate)
        
        # Phase 1: Shuffle and segment into mini-batches
        mini_batches = create_mini_batches(X_train, y_train, batch_size, seed=epoch)
        epoch_cost = 0
        
        for X_batch, y_batch in mini_batches:
            t += 1 # Increment Adam step counter
            
            # Phase 3: Data Augmentation
            X_batch_aug = apply_data_augmentation(X_batch)
            
            # Phase 3: Forward propagation with active Dropout
            AL, caches = forward_propagation_dropout(X_batch_aug, parameters, keep_prob)
            
            # Phase 3: Regularized Loss Calculation (L2)
            batch_cost = compute_cost_l2(AL, y_batch, parameters, lambd)
            epoch_cost += batch_cost * (X_batch.shape[0] / X_train.shape[0])
            
            # Phase 3: Backpropagation with integrated adjustments
            grads = backward_propagation_reg(caches, parameters, y_batch, lambd, keep_prob)
            
            # Phase 4: Turbocharged update via Adam
            parameters, m, v = update_parameters_adam(parameters, grads, m, v, t, current_lr)
            
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:02d} | Dynamic LR: {current_lr:.5f} | Turbo Cost: {epoch_cost:.4f}")
            
    return parameters

In [84]:
if __name__ == "__main__":
    print("--- Running Phase 4: Testing the Turbocharged Pipeline ---")
    np.random.seed(42)
    
    # Generate mock training dataset (5000 images flattened down to 100 features)
    X_train_final = np.random.randn(5000, 100)
    y_train_final = np.eye(4)[np.random.randint(0, 4, size=(5000,))] # 4 structural output classes
    
    network_layout = [100, 32, 16, 4] # Deep layout
    
    # Run the completed system
    final_model_parameters = train_model_turbocharged(
        X_train_final, y_train_final,
        layer_dims=network_layout,
        epochs=40,
        batch_size=128,
        initial_lr=0.01,
        lambd=0.2,
        keep_prob=0.85,
        decay_rate=0.1
    )
    print("\n🎉 Congratulations! Your modular machine learning pipeline is fully built and functioning!")

--- Running Phase 4: Testing the Turbocharged Pipeline ---
Epoch 00 | Dynamic LR: 0.01000 | Turbo Cost: 1.6116
Epoch 10 | Dynamic LR: 0.00500 | Turbo Cost: 1.3525
Epoch 20 | Dynamic LR: 0.00333 | Turbo Cost: 1.2743
Epoch 30 | Dynamic LR: 0.00250 | Turbo Cost: 1.2246
Epoch 39 | Dynamic LR: 0.00204 | Turbo Cost: 1.1959

🎉 Congratulations! Your modular machine learning pipeline is fully built and functioning!


In [86]:
import urllib.request
import tarfile
import pickle
import os
import numpy as np

def load_cifar10_dataset(target_dir="./cifar10_data"):
    """
    Automatically downloads, extracts, and parses the raw CIFAR-10 dataset
    into flat, normalized NumPy arrays ready for our custom network layers.
    """
    url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    archive_path = os.path.join(target_dir, "cifar-10-python.tar.gz")
    
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
        
    # Step 1: Download raw binary data from source if it doesn't exist
    if not os.path.exists(archive_path):
        print("Downloading official CIFAR-10 dataset (approx 170MB)... Please wait.")
        urllib.request.urlretrieve(url, archive_path)
        print("Download complete.")
        
    # Step 2: Extract tar file archive
    extracted_folder = os.path.join(target_dir, "cifar-10-batches-py")
    if not os.path.exists(extracted_folder):
        print("Extracting archive files...")
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(path=target_dir)
        print("Extraction complete.")
            
    # Step 3: Parse and load data batches
    def load_batch(file_path):
        with open(file_path, 'rb') as f:
            batch_dict = pickle.load(f, encoding='bytes')
        X = batch_dict[b'data']  # shape: (10000, 3072)
        Y = np.array(batch_dict[b'labels'])  # shape: (10000,)
        return X, Y

    print("Loading data into memory...")
    # Load batch 1 for training data (10,000 images is perfect for fast local CPU training)
    X_raw, Y_raw = load_batch(os.path.join(extracted_folder, "data_batch_1"))
    
    # Step 4: Vectorized preprocessing
    # Standard normalization: scale pixels down from [0, 255] to [0.0, 1.0] range
    X_processed = X_raw.astype(float) / 255.0
    
    # One-hot encode the labels (10 distinct visual object classes)
    num_classes = 10
    Y_one_hot = np.eye(num_classes)[Y_raw]
    
    print(f"Dataset successfully loaded. Image matrix shape: {X_processed.shape} | Labels matrix shape: {Y_one_hot.shape}")
    return X_processed, Y_one_hot

In [89]:
if __name__ == "__main__":
    print("--- 🚀 STAGE 5: RUNNING TURBOCHARGED ENGINE ON REAL IMAGES 🚀 ---")
    
    # 1. Fetch real image array structures from CIFAR-10
    X_all, y_all = load_cifar10_dataset()
    
    # 2. Phase 1: Split data using our 98% / 1% / 1% Modern Split Logic
    X_train, y_train, X_dev, y_dev, X_test, y_test = prepare_data_splits(
        X_all, y_all, train_prop=0.98, dev_prop=0.01
    )
    
    # 3. Phase 1: Apply statistical feature normalization across the data splits
    X_train, X_dev, X_test = normalization_features(X_train, X_dev, X_test)
    
    # 4. Define Network Layout: 3072 pixel inputs -> 128 hidden -> 64 hidden -> 10 output classes
    network_layout = [3072, 128, 64, 10]
    
    # 5. Execute optimization using all built concepts
    print("\nTraining on real images initialized...")
    final_parameters = train_model_turbocharged(
        X_train, y_train,
        layer_dims=network_layout,
        epochs=30,
        batch_size=256,   # Power of 2 mini-batch sizes
        initial_lr=0.01,  # Base scale
        lambd=0.5,        # L2 penalty parameter
        keep_prob=0.8,    # Dropout layer threshold
        decay_rate=0.1    # Dynamic learning rate decay factor
    )
    
    print("\n🎉 Process complete! The pipeline has successfully ingested, stabilized, regularized, and optimized an image classifier on real data.")

--- 🚀 STAGE 5: RUNNING TURBOCHARGED ENGINE ON REAL IMAGES 🚀 ---
Loading data into memory...
Dataset successfully loaded. Image matrix shape: (10000, 3072) | Labels matrix shape: (10000, 10)

Training on real images initialized...
Epoch 00 | Dynamic LR: 0.01000 | Turbo Cost: 4.1928
Epoch 10 | Dynamic LR: 0.00500 | Turbo Cost: 1.9995
Epoch 20 | Dynamic LR: 0.00333 | Turbo Cost: 1.8574
Epoch 29 | Dynamic LR: 0.00256 | Turbo Cost: 1.7410

🎉 Process complete! The pipeline has successfully ingested, stabilized, regularized, and optimized an image classifier on real data.


In [90]:
## evaluating on real world dataset
def evaluate_model(X, Y, parameters):
    """
    Computes accuracy and loss for a dataset split.
    """
    # Run a clean forward pass (keep_prob=1.0 because we NEVER use dropout during evaluation)
    AL, _ = forward_propagation_dropout(X, parameters, keep_prob=1.0)
    
    # Calculate Cross-Entropy Loss (lambd=0 because we don't penalize evaluation tracking)
    loss = compute_cost_l2(AL, Y, parameters, lambd=0)
    
    # Vectorized Accuracy calculation
    predictions = np.argmax(AL, axis=0)
    true_labels = np.argmax(Y.T, axis=0)
    accuracy = np.mean(predictions == true_labels) * 100
    
    return accuracy, loss

In [91]:
if __name__ == "__main__":
    # ... Keep all your data loading and train_model_turbocharged code here ...
    
    # 1. Run the evaluation function across all three data splits
    train_acc, train_loss = evaluate_model(X_train, y_train, final_parameters)
    dev_acc, dev_loss     = evaluate_model(X_dev, y_dev, final_parameters)
    test_acc, test_loss   = evaluate_model(X_test, y_test, final_parameters)
    
    # 2. Print the final dashboard
    print("\n================ EVALUATION DASHBOARD ================")
    print(f"🥇 Training Set   -> Accuracy: {train_acc:.2f}% | Loss: {train_loss:.4f}")
    print(f"🥈 Dev Set (Val)  -> Accuracy: {dev_acc:.2f}% | Loss: {dev_loss:.4f}")
    print(f"🥉 Test Set (New) -> Accuracy: {test_acc:.2f}% | Loss: {test_loss:.4f}")
    print("======================================================")


================ EVALUATION DASHBOARD ================
🥇 Training Set   -> Accuracy: 52.01% | Loss: 1.3720
🥈 Dev Set (Val)  -> Accuracy: 43.00% | Loss: 1.6418
🥉 Test Set (New) -> Accuracy: 42.00% | Loss: 1.5367
